In [40]:
import pandas as pd
import numpy as np

In [41]:
from sklearn.impute import SimpleImputer #because fever  column contain missing value
from sklearn.preprocessing import OneHotEncoder  #for gender,city
from sklearn.preprocessing import OrdinalEncoder #for cough 

In [42]:
df=pd.read_csv("C:/Users/Dell/Downloads/covid_toy.csv")

In [43]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [44]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [45]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [46]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [47]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)


In [48]:
X_train

,age,gender,fever,cough,city
85,16,Female,103.0,Mild,Bangalore
13,64,Male,102.0,Mild,Bangalore
21,73,Male,98.0,Mild,Bangalore
52,47,Female,100.0,Strong,Bangalore
82,24,Male,98.0,Mild,Kolkata
...,...,...,...,...,...
62,56,Female,104.0,Strong,Bangalore
83,17,Female,104.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
35,82,Female,102.0,Strong,Bangalore


# without column transformer

In [49]:
#adding simple imputer to fever col
si=SimpleImputer()
X_train_fever=si.fit_transform(X_train[['fever']])

#also the test dat
X_test_fever=si.fit_transform(X_test[['fever']])

In [50]:
X_train_fever.shape

(80, 1)

In [51]:
#Ordinalencoding ->cough
oe=OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough=oe.fit_transform(X_train[['cough']])

X_test_cough=oe.fit_transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [60]:
#OnehotEncoding ->geneder,city
ohe=OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city=ohe.fit_transform(X_train[['gender','city']])

X_test_gender_city=ohe.transform(X_test[['gender','city']])
X_train_gender_city.shape

(80, 4)

In [61]:
#Extracting Age
X_train_age=X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age=X_test.drop(columns=['gender','fever','cough','city']).values
X_train_age.shape

(80, 1)

In [64]:
X_train_transformed=np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
X_train_transformed.shape

(80, 7)

# with help of columnTransformer

In [65]:
from sklearn.compose import ColumnTransformer

In [71]:
transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [73]:
transformer.fit_transform(X_train).shape

(80, 7)

In [74]:
transformer.fit_transform(X_test).shape

(20, 7)